# How to efficiently query properties

In the following we presents first how `StructureData` nodes are store in the AiiDA database and then how to user the `QueryBuilder` to find them.

## How properties are store in the AiiDA database

In the AiiDA database, we store only properties which are defined to be different from the default value. For example, if we provide all the charges to be zero, the `charges` properties will not be stored in the database. This means that the only structures which are in the database and contains also a charges entry, will be about systems with charged different from zero (NB can also be overall neutral).

For a given structure, you can print the list of all the properties stored in the database (and so, queryable), by calling the `get_defined_properties` method and providing `exclude_computed=False` as input (i.e. returning also properties which are computed/derived from the user-defined ones):

In [ ]:
from aiida import load_profile
from aiida_atomistic import StructureData

load_profile()

structure = StructureData(**{
    'pbc': [True, True, True],
    'cell': [[2.75, 2.75, 0.0], [0.0, 2.75, 2.75], [2.75, 0.0, 2.75]],
    'symbols': ['Si', 'Si'],
    'charges': [0.0, 0.0],
    'kinds': ['Si', 'Si'],
    'positions': [
        [0.0, 0.0, 0.0],
        [3.84, 1.3576450198781713, 1.9200]
    ],
})

print("Properties stored in the database are:")
print(structure.get_defined_properties(exclude_computed=False))

{'positions', 'dimensionality', 'symbols', 'cell_volume', 'cell', 'kinds', 'masses', 'formula', 'sites'}


Where actually the only properties stored in the AiiDA database are:

In [12]:
print(structure.get_defined_properties())

{'positions', 'symbols', 'cell', 'kinds', 'masses'}


The only exception will be the `sites` property, which is contained in this list but not stored in the database: this is computed on-the-fly and it does not contains additional information meant to be in the database.

:::{note} 
:class: dropdown
To explicitely see how data are stored and represented in the database, you can access the `structure.base.attributes.all` dictionary. 
As you can see, not `sites` entry is present.

```python
print(structure.base.attributes.all.keys())
```

returns: 

```bash
dict_keys(['cell', 'symbols', 'positions', 'kinds', 'masses', 'cell_volume', 'dimensionality', 'formula'])
```
:::

## How to Query StructureData using properties

In the following we present how to perform advanced query in you database to retrieve `StructureData` with given properties.

Thanks to the additional computed properties in our `StructureData` (*formula*, *symbols*, *kinds*, *masses*, *charges*, *magmoms*, *positions*, *cell_volume*, *dimensionality*), we can easily query for a structure:

In [ ]:
from aiida.orm import QueryBuilder

stored = StructureData.from_mutable(mutable_structure).store()
print(stored.pk)

qb = QueryBuilder()
qb.append(StructureData, 
          filters={'attributes.formula': 'Fe2'},
          )

qb.all()